# RUN ANCOM-BC2 on the metagenomic AD data

In [4]:
setwd("~/scratch/AD_EM")

library("ANCOMBC")
library("tidyverse")
library("data.table")
#library("DT")
#library("phyloseq")

In [5]:
#load data
md<-fread("metagenomics/data_files/overly_specified_metadata.tsv")%>%
  mutate(fasted=ifelse(zt_time<20 & zt_time>4, "fasted","fed"),
         mouseid = sub("\\..*", "",sample_name),
         genotype_diet = factor(genotype_diet, levels = c("Ntg.ALF", "Tg.ALF", "Tg.TRF")))%>%
  column_to_rownames("sample_name")

feattab<-fread("metagenomics/data_files/159013_wolR2_per_genome.feature-table.species.tsv")%>%
  dplyr::rename(taxonomy=`OTU ID`)%>%
  rename_with(~str_replace_all(.,"^[^.]*\\.", ""))%>%
  column_to_rownames("taxonomy")%>%
  dplyr::select(any_of(rownames(md)))%>%as.matrix()

In [6]:
#run ANCOM-BC2
set.seed(123)
output = ancombc2(data = feattab,
                  aggregate_data = feattab,
                  meta_data = md,
                  fix_formula = "genotype_diet*zt_time",
                  rand_formula = "(zt_time | mouseid)",
                  p_adj_method = "fdr", pseudo_sens = TRUE,
                  prv_cut = 0.10, lib_cut = 1000, s0_perc = 0.05,
                  group = "genotype_diet", struc_zero = TRUE, neg_lb = TRUE,
                  alpha = 0.05, n_cl = 2, verbose = TRUE,
                  global = TRUE, pairwise = TRUE, dunnet = TRUE, trend = TRUE,
                  iter_control = list(tol = 1e-2, max_iter = 20, 
                                      verbose = TRUE),
                  em_control = list(tol = 1e-5, max_iter = 100),
                  lme_control = lme4::lmerControl(),
                  mdfdr_control = list(fwer_ctrl_method = "fdr", B = 100),
                  trend_control = list(contrast = list(matrix(c(1, 0, -1, 1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(-1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE)),
                                       node = list(2, 2, 1),
                                       solver = "ECOS",
                                       B = 10))

Checking the input data type ...

The input data is of type: matrix

The imported data is in a generic 'matrix'/'data.frame' format.

PASS

Checking the sample metadata ...

The specified variables in the formula: genotype_diet, zt_time

The available variables in the sample metadata: sex, mouse_genotype, feeding_condition, zt_collection_time, zt_time, phase, exp_condition, genotype_diet, food_availability, food_condition, sex_condition, sex_zt, sex_exp_condition, sex_cond_fed, fasted, mouseid

PASS

Checking other arguments ...

The number of groups of interest is: 3

The sample size per group is: Ntg.ALF = 48, Tg.ALF = 53, Tg.TRF = 44

PASS

Obtaining initial estimates ...

boundary (singular) fit: see help('isSingular')

REML iteration = 1, epsilon = 4

REML iteration = 2, epsilon = 13

REML iteration = 3, epsilon = 1.6

REML iteration = 4, epsilon = 1.1

REML iteration = 5, epsilon = 0.55

REML iteration = 6, epsilon = 0.58

REML iteration = 7, epsilon = 0.33

REML iteration = 8, e

In [10]:
res_prim = output$res
write.table(res_prim,"metagenomics/ancombc/ancombbc_primaryres_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [11]:
res_global = output$res_global 
write.table(res_global,"metagenomics/ancombc/ancombbc_globalres_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [34]:
res_pair = output$res_pair 
write.table(res_pair,"metagenomics/ancombc/ancombbc_pairwisecomparres_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [13]:
res_dunn = output$res_dunn
write.table(res_dunn,"metagenomics/ancombc/ancombbc_dunnett_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [14]:
res_trend = output$res_trend
write.table(res_trend,"metagenomics/ancombc/ancombbc_pattern_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [21]:
colnames(res_prim)

[1] "taxon"                                
 [2] "lfc_(Intercept)"                      
 [3] "lfc_genotype_dietTg.ALF"              
 [4] "lfc_genotype_dietTg.TRF"              
 [5] "lfc_zt_time"                          
 [6] "lfc_genotype_dietTg.ALF:zt_time"      
 [7] "lfc_genotype_dietTg.TRF:zt_time"      
 [8] "se_(Intercept)"                       
 [9] "se_genotype_dietTg.ALF"               
[10] "se_genotype_dietTg.TRF"               
[11] "se_zt_time"                           
[12] "se_genotype_dietTg.ALF:zt_time"       
[13] "se_genotype_dietTg.TRF:zt_time"       
[14] "W_(Intercept)"                        
[15] "W_genotype_dietTg.ALF"                
[16] "W_genotype_dietTg.TRF"                
[17] "W_zt_time"                            
[18] "W_genotype_dietTg.ALF:zt_time"        
[19] "W_genotype_dietTg.TRF:zt_time"        
[20] "p_(Intercept)"                        
[21] "p_genotype_dietTg.ALF"                
[22] "p_genotype_dietTg.TRF"                
[23] "p_zt_time"                            
[24] "p_genotype_dietTg.ALF:zt_time"        
[25] "p_genotype_dietTg.TRF:zt_time"        
[26] "q_(Intercept)"                        
[27] "q_genotype_dietTg.ALF"                
[28] "q_genotype_dietTg.TRF"                
[29] "q_zt_time"                            
[30] "q_genotype_dietTg.ALF:zt_time"        
[31] "q_genotype_dietTg.TRF:zt_time"        
[32] "diff_(Intercept)"                     
[33] "diff_genotype_dietTg.ALF"             
[34] "diff_genotype_dietTg.TRF"             
[35] "diff_zt_time"                         
[36] "diff_genotype_dietTg.ALF:zt_time"     
[37] "diff_genotype_dietTg.TRF:zt_time"     
[38] "passed_ss_(Intercept)"                
[39] "passed_ss_genotype_dietTg.ALF"        
[40] "passed_ss_genotype_dietTg.TRF"        
[41] "passed_ss_zt_time"                    
[42] "passed_ss_genotype_dietTg.ALF:zt_time"
[43] "passed_ss_genotype_dietTg.TRF:zt_time"

## Running but using a different reference (Tg-ALF)

In [35]:
#load data
md<-fread("metagenomics/data_files/overly_specified_metadata.tsv")%>%
  mutate(fasted=ifelse(zt_time<20 & zt_time>4, "fasted","fed"),
         mouseid = sub("\\..*", "",sample_name),
         genotype_diet = factor(genotype_diet, levels = c("Tg.ALF", "Ntg.ALF","Tg.TRF")))%>%
  column_to_rownames("sample_name")

feattab<-fread("metagenomics/data_files/159013_wolR2_per_genome.feature-table.species.tsv")%>%
  dplyr::rename(taxonomy=`OTU ID`)%>%
  rename_with(~str_replace_all(.,"^[^.]*\\.", ""))%>%
  column_to_rownames("taxonomy")%>%
  dplyr::select(any_of(rownames(md)))%>%as.matrix()

In [36]:
#run ANCOM-BC2
set.seed(123)
output_trf = ancombc2(data = feattab,
                  aggregate_data = feattab,
                  meta_data = md,
                  fix_formula = "genotype_diet*zt_time",
                  rand_formula = "(zt_time | mouseid)",
                  p_adj_method = "fdr", pseudo_sens = TRUE,
                  prv_cut = 0.10, lib_cut = 1000, s0_perc = 0.05,
                  group = "genotype_diet", struc_zero = TRUE, neg_lb = TRUE,
                  alpha = 0.05, n_cl = 2, verbose = TRUE,
                  global = TRUE, pairwise = TRUE, dunnet = TRUE, trend = TRUE,
                  iter_control = list(tol = 1e-2, max_iter = 20, 
                                      verbose = TRUE),
                  em_control = list(tol = 1e-5, max_iter = 100),
                  lme_control = lme4::lmerControl(),
                  mdfdr_control = list(fwer_ctrl_method = "fdr", B = 100),
                  trend_control = list(contrast = list(matrix(c(1, 0, -1, 1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(-1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE)),
                                       node = list(2, 2, 1),
                                       solver = "ECOS",
                                       B = 10))

Checking the input data type ...

The input data is of type: matrix

The imported data is in a generic 'matrix'/'data.frame' format.

PASS

Checking the sample metadata ...

The specified variables in the formula: genotype_diet, zt_time

The available variables in the sample metadata: sex, mouse_genotype, feeding_condition, zt_collection_time, zt_time, phase, exp_condition, genotype_diet, food_availability, food_condition, sex_condition, sex_zt, sex_exp_condition, sex_cond_fed, fasted, mouseid

PASS

Checking other arguments ...

The number of groups of interest is: 3

The sample size per group is: Tg.ALF = 53, Ntg.ALF = 48, Tg.TRF = 44

PASS

Obtaining initial estimates ...

boundary (singular) fit: see help('isSingular')

REML iteration = 1, epsilon = 4

REML iteration = 2, epsilon = 12

REML iteration = 3, epsilon = 1.7

REML iteration = 4, epsilon = 1.6

REML iteration = 5, epsilon = 0.54

REML iteration = 6, epsilon = 0.54

REML iteration = 7, epsilon = 0.31

REML iteration = 8, e

In [37]:
res_prim = output_trf$res
write.table(res_prim,"metagenomics/ancombc/ancombbc_primaryrestrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [38]:
res_global = output_trf$res_global 
write.table(res_global,"metagenomics/ancombc/ancombbc_globalrestrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [39]:
res_pair = output_trf$res_pair 
write.table(res_pair,"metagenomics/ancombc/ancombbc_pairwisecomparrestrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [40]:
res_dunn = output_trf$res_dunn
write.table(res_dunn,"metagenomics/ancombc/ancombbc_dunnetttrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [41]:
res_trend = output_trf$res_trend
write.table(res_trend,"metagenomics/ancombc/ancombbc_patterntrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

## Running but using fasted/fed variable instead of ZT

In [44]:
#load data
md<-fread("metagenomics/data_files/overly_specified_metadata.tsv")%>%
  mutate(fasted=ifelse(zt_time<20 & zt_time>4, "fasted","fed"),
         mouseid = sub("\\..*", "",sample_name),
         genotype_diet = factor(genotype_diet, levels = c("Ntg.ALF","Tg.ALF","Tg.TRF")))%>%
  column_to_rownames("sample_name")

feattab<-fread("metagenomics/data_files/159013_wolR2_per_genome.feature-table.species.tsv")%>%
  dplyr::rename(taxonomy=`OTU ID`)%>%
  rename_with(~str_replace_all(.,"^[^.]*\\.", ""))%>%
  column_to_rownames("taxonomy")%>%
  dplyr::select(any_of(rownames(md)))%>%as.matrix()

In [45]:
#run ANCOM-BC2
set.seed(123)
output_fasted = ancombc2(data = feattab,
                  aggregate_data = feattab,
                  meta_data = md,
                  fix_formula = "genotype_diet*fasted",
                  rand_formula = "(1| mouseid)",
                  p_adj_method = "fdr", pseudo_sens = TRUE,
                  prv_cut = 0.10, lib_cut = 1000, s0_perc = 0.05,
                  group = "genotype_diet", struc_zero = TRUE, neg_lb = TRUE,
                  alpha = 0.05, n_cl = 2, verbose = TRUE,
                  global = TRUE, pairwise = TRUE, dunnet = TRUE, trend = TRUE,
                  iter_control = list(tol = 1e-2, max_iter = 20, 
                                      verbose = TRUE),
                  em_control = list(tol = 1e-5, max_iter = 100),
                  lme_control = lme4::lmerControl(),
                  mdfdr_control = list(fwer_ctrl_method = "fdr", B = 100),
                  trend_control = list(contrast = list(matrix(c(1, 0, -1, 1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(-1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE)),
                                       node = list(2, 2, 1),
                                       solver = "ECOS",
                                       B = 10))

Checking the input data type ...

The input data is of type: matrix

The imported data is in a generic 'matrix'/'data.frame' format.

PASS

Checking the sample metadata ...

The specified variables in the formula: genotype_diet, fasted

The available variables in the sample metadata: sex, mouse_genotype, feeding_condition, zt_collection_time, zt_time, phase, exp_condition, genotype_diet, food_availability, food_condition, sex_condition, sex_zt, sex_exp_condition, sex_cond_fed, fasted, mouseid

PASS

Checking other arguments ...

The number of groups of interest is: 3

The sample size per group is: Ntg.ALF = 48, Tg.ALF = 53, Tg.TRF = 44

PASS

Obtaining initial estimates ...

boundary (singular) fit: see help('isSingular')

REML iteration = 1, epsilon = 4

REML iteration = 2, epsilon = 18

REML iteration = 3, epsilon = 2.3

REML iteration = 4, epsilon = 1.3

REML iteration = 5, epsilon = 1

REML iteration = 6, epsilon = 0.6

REML iteration = 7, epsilon = 0.89

REML iteration = 8, epsilo

In [46]:
res_prim = output_fasted$res
write.table(res_prim,"metagenomics/ancombc/ancombbc_primaryresfasted_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [47]:
res_global = output_fasted$res_global 
write.table(res_global,"metagenomics/ancombc/ancombbc_globalresfasted_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [48]:
res_pair = output_fasted$res_pair 
write.table(res_pair,"metagenomics/ancombc/ancombbc_pairwisecomparresfasted_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [49]:
res_dunn = output_fasted$res_dunn
write.table(res_dunn,"metagenomics/ancombc/ancombbc_dunnettfasted_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [50]:
res_trend = output_fasted$res_trend
write.table(res_trend,"metagenomics/ancombc/ancombbc_patternfasted_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

## Running but using fasted/fed variable instead of ZT (using diff reference)

In [51]:
#load data
md<-fread("metagenomics/data_files/overly_specified_metadata.tsv")%>%
  mutate(fasted=ifelse(zt_time<20 & zt_time>4, "fasted","fed"),
         mouseid = sub("\\..*", "",sample_name),
         genotype_diet = factor(genotype_diet, levels = c("Tg.ALF","Ntg.ALF","Tg.TRF")))%>%
  column_to_rownames("sample_name")

feattab<-fread("metagenomics/data_files/159013_wolR2_per_genome.feature-table.species.tsv")%>%
  dplyr::rename(taxonomy=`OTU ID`)%>%
  rename_with(~str_replace_all(.,"^[^.]*\\.", ""))%>%
  column_to_rownames("taxonomy")%>%
  dplyr::select(any_of(rownames(md)))%>%as.matrix()

In [52]:
#run ANCOM-BC2
set.seed(123)
output_fastedtrf = ancombc2(data = feattab,
                  aggregate_data = feattab,
                  meta_data = md,
                  fix_formula = "genotype_diet*fasted",
                  rand_formula = "(1| mouseid)",
                  p_adj_method = "fdr", pseudo_sens = TRUE,
                  prv_cut = 0.10, lib_cut = 1000, s0_perc = 0.05,
                  group = "genotype_diet", struc_zero = TRUE, neg_lb = TRUE,
                  alpha = 0.05, n_cl = 2, verbose = TRUE,
                  global = TRUE, pairwise = TRUE, dunnet = TRUE, trend = TRUE,
                  iter_control = list(tol = 1e-2, max_iter = 20, 
                                      verbose = TRUE),
                  em_control = list(tol = 1e-5, max_iter = 100),
                  lme_control = lme4::lmerControl(),
                  mdfdr_control = list(fwer_ctrl_method = "fdr", B = 100),
                  trend_control = list(contrast = list(matrix(c(1, 0, -1, 1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(-1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE)),
                                       node = list(2, 2, 1),
                                       solver = "ECOS",
                                       B = 10))

Checking the input data type ...

The input data is of type: matrix

The imported data is in a generic 'matrix'/'data.frame' format.

PASS

Checking the sample metadata ...

The specified variables in the formula: genotype_diet, fasted

The available variables in the sample metadata: sex, mouse_genotype, feeding_condition, zt_collection_time, zt_time, phase, exp_condition, genotype_diet, food_availability, food_condition, sex_condition, sex_zt, sex_exp_condition, sex_cond_fed, fasted, mouseid

PASS

Checking other arguments ...

The number of groups of interest is: 3

The sample size per group is: Tg.ALF = 53, Ntg.ALF = 48, Tg.TRF = 44

PASS

Obtaining initial estimates ...

boundary (singular) fit: see help('isSingular')

REML iteration = 1, epsilon = 4

REML iteration = 2, epsilon = 22

REML iteration = 3, epsilon = 3.1

REML iteration = 4, epsilon = 1.3

REML iteration = 5, epsilon = 0.94

REML iteration = 6, epsilon = 0.58

REML iteration = 7, epsilon = 0.46

REML iteration = 8, ep

In [53]:
res_prim = output_fastedtrf$res
write.table(res_prim,"metagenomics/ancombc/ancombbc_primaryresfastedtrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [54]:
res_global = output_fastedtrf$res_global 
write.table(res_global,"metagenomics/ancombc/ancombbc_globalresfastedtrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [55]:
res_pair = output_fastedtrf$res_pair 
write.table(res_pair,"metagenomics/ancombc/ancombbc_pairwisecomparresfastedtrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [56]:
res_dunn = output_fastedtrf$res_dunn
write.table(res_dunn,"metagenomics/ancombc/ancombbc_dunnettfastedtrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [57]:
res_trend = output_fastedtrf$res_trend
write.table(res_trend,"metagenomics/ancombc/ancombbc_patternfastedtrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

## Running but using light/dark variable instead of ZT (using diff reference)

In [61]:
#load data
md<-fread("metagenomics/data_files/overly_specified_metadata.tsv")%>%
  mutate(fasted=ifelse(zt_time<20 & zt_time>4, "fasted","fed"),
         diurnal=ifelse(zt_time<12,"light","dark"),
         mouseid = sub("\\..*", "",sample_name),
         genotype_diet = factor(genotype_diet, levels = c("Tg.ALF","Ntg.ALF","Tg.TRF")))%>%
  column_to_rownames("sample_name")

feattab<-fread("metagenomics/data_files/159013_wolR2_per_genome.feature-table.species.tsv")%>%
  dplyr::rename(taxonomy=`OTU ID`)%>%
  rename_with(~str_replace_all(.,"^[^.]*\\.", ""))%>%
  column_to_rownames("taxonomy")%>%
  dplyr::select(any_of(rownames(md)))%>%as.matrix()

In [62]:
#run ANCOM-BC2
set.seed(123)
output_lighttrf = ancombc2(data = feattab,
                  aggregate_data = feattab,
                  meta_data = md,
                  fix_formula = "genotype_diet*diurnal",
                  rand_formula = "(1| mouseid)",
                  p_adj_method = "fdr", pseudo_sens = TRUE,
                  prv_cut = 0.10, lib_cut = 1000, s0_perc = 0.05,
                  group = "genotype_diet", struc_zero = TRUE, neg_lb = TRUE,
                  alpha = 0.05, n_cl = 2, verbose = TRUE,
                  global = TRUE, pairwise = TRUE, dunnet = TRUE, trend = TRUE,
                  iter_control = list(tol = 1e-2, max_iter = 20, 
                                      verbose = TRUE),
                  em_control = list(tol = 1e-5, max_iter = 100),
                  lme_control = lme4::lmerControl(),
                  mdfdr_control = list(fwer_ctrl_method = "fdr", B = 100),
                  trend_control = list(contrast = list(matrix(c(1, 0, -1, 1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(-1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE)),
                                       node = list(2, 2, 1),
                                       solver = "ECOS",
                                       B = 10))

Checking the input data type ...

The input data is of type: matrix

The imported data is in a generic 'matrix'/'data.frame' format.

PASS

Checking the sample metadata ...

The specified variables in the formula: genotype_diet, diurnal

The available variables in the sample metadata: sex, mouse_genotype, feeding_condition, zt_collection_time, zt_time, phase, exp_condition, genotype_diet, food_availability, food_condition, sex_condition, sex_zt, sex_exp_condition, sex_cond_fed, fasted, diurnal, mouseid

PASS

Checking other arguments ...

The number of groups of interest is: 3

The sample size per group is: Tg.ALF = 53, Ntg.ALF = 48, Tg.TRF = 44

PASS

Obtaining initial estimates ...

boundary (singular) fit: see help('isSingular')

REML iteration = 1, epsilon = 4

REML iteration = 2, epsilon = 22

REML iteration = 3, epsilon = 3.4

REML iteration = 4, epsilon = 1.3

REML iteration = 5, epsilon = 0.8

REML iteration = 6, epsilon = 0.55

REML iteration = 7, epsilon = 0.39

REML iteratio

In [63]:
res_prim = output_lighttrf$res
write.table(res_prim,"metagenomics/ancombc/ancombbc_primaryreslighttrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [64]:
res_global = output_lighttrf$res_global 
write.table(res_global,"metagenomics/ancombc/ancombbc_globalreslighttrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [65]:
res_pair = output_lighttrf$res_pair 
write.table(res_pair,"metagenomics/ancombc/ancombbc_pairwisecomparreslighttrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [66]:
res_dunn = output_lighttrf$res_dunn
write.table(res_dunn,"metagenomics/ancombc/ancombbc_dunnettlighttrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [67]:
res_trend = output_lighttrf$res_trend
write.table(res_trend,"metagenomics/ancombc/ancombbc_patternlighttrf_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

## Running but using light/dark variable instead (shifting one) of ZT (using diff reference)

In [71]:
#load data
md<-fread("metagenomics/data_files/overly_specified_metadata.tsv")%>%
  mutate(fasted=ifelse(zt_time<20 & zt_time>4, "fasted","fed"),
         diurnal=ifelse(zt_time<13 & zt_time>0,"light","dark"),
         mouseid = sub("\\..*", "",sample_name),
         genotype_diet = factor(genotype_diet, levels = c("Tg.ALF","Ntg.ALF","Tg.TRF")))%>%
  column_to_rownames("sample_name")

feattab<-fread("metagenomics/data_files/159013_wolR2_per_genome.feature-table.species.tsv")%>%
  dplyr::rename(taxonomy=`OTU ID`)%>%
  rename_with(~str_replace_all(.,"^[^.]*\\.", ""))%>%
  column_to_rownames("taxonomy")%>%
  dplyr::select(any_of(rownames(md)))%>%as.matrix()

In [72]:
#run ANCOM-BC2
set.seed(123)
output_lighttrf2 = ancombc2(data = feattab,
                  aggregate_data = feattab,
                  meta_data = md,
                  fix_formula = "genotype_diet*diurnal",
                  rand_formula = "(1| mouseid)",
                  p_adj_method = "fdr", pseudo_sens = TRUE,
                  prv_cut = 0.10, lib_cut = 1000, s0_perc = 0.05,
                  group = "genotype_diet", struc_zero = TRUE, neg_lb = TRUE,
                  alpha = 0.05, n_cl = 2, verbose = TRUE,
                  global = TRUE, pairwise = TRUE, dunnet = TRUE, trend = TRUE,
                  iter_control = list(tol = 1e-2, max_iter = 20, 
                                      verbose = TRUE),
                  em_control = list(tol = 1e-5, max_iter = 100),
                  lme_control = lme4::lmerControl(),
                  mdfdr_control = list(fwer_ctrl_method = "fdr", B = 100),
                  trend_control = list(contrast = list(matrix(c(1, 0, -1, 1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(-1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE),
                                                       matrix(c(1, 0, 1, -1),
                                                              nrow = 2, 
                                                              byrow = TRUE)),
                                       node = list(2, 2, 1),
                                       solver = "ECOS",
                                       B = 10))

Checking the input data type ...

The input data is of type: matrix

The imported data is in a generic 'matrix'/'data.frame' format.

PASS

Checking the sample metadata ...

The specified variables in the formula: genotype_diet, diurnal

The available variables in the sample metadata: sex, mouse_genotype, feeding_condition, zt_collection_time, zt_time, phase, exp_condition, genotype_diet, food_availability, food_condition, sex_condition, sex_zt, sex_exp_condition, sex_cond_fed, fasted, diurnal, mouseid

PASS

Checking other arguments ...

The number of groups of interest is: 3

The sample size per group is: Tg.ALF = 53, Ntg.ALF = 48, Tg.TRF = 44

PASS

Obtaining initial estimates ...

REML iteration = 1, epsilon = 3.9

REML iteration = 2, epsilon = 27

REML iteration = 3, epsilon = 4.1

REML iteration = 4, epsilon = 1.5

REML iteration = 5, epsilon = 0.86

REML iteration = 6, epsilon = 0.59

REML iteration = 7, epsilon = 0.41

REML iteration = 8, epsilon = 0.29

REML iteration = 9, eps

In [73]:
res_prim = output_lighttrf2$res
write.table(res_prim,"metagenomics/ancombc/ancombbc_primaryreslighttrf2_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [74]:
res_global = output_lighttrf2$res_global 
write.table(res_global,"metagenomics/ancombc/ancombbc_globalreslighttrf2_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [75]:
res_pair = output_lighttrf2$res_pair 
write.table(res_pair,"metagenomics/ancombc/ancombbc_pairwisecomparreslighttrf2_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [78]:
res_dunn = output_lighttrf2$res_dunn
write.table(res_dunn,"metagenomics/ancombc/ancombbc_dunnettlighttrf2_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [79]:
res_trend = output_lighttrf2$res_trend
write.table(res_trend,"metagenomics/ancombc/ancombbc_patternlighttrf2_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

## looking for mci differences on human BEAM

In [90]:
#load data
md<-fread("metagenomics/human_BEAM/qiita_sample_metadata.preab.tsv")%>%
         mutate(memdx = factor(memdx, levels = c("nc","mci")))%>%
  column_to_rownames("sample_name")

feattab<-fread("metagenomics/human_BEAM/199894_none.cov20.clr.csv")%>%
  dplyr::rename(taxonomy=V1)%>%
  column_to_rownames("taxonomy")%>%
  dplyr::select(any_of(rownames(md)))%>%as.matrix()

In [108]:
#run ANCOM-BC2
set.seed(123)
output_human = ancombc2(data = feattab,
                  aggregate_data = feattab,
                  meta_data = md,
                  fix_formula = "memdx",
                  rand_formula = NULL,
                  p_adj_method = "fdr", pseudo_sens = FALSE,
                  prv_cut = 0.10, lib_cut = 0, s0_perc = 0.05,
                  group = "memdx", struc_zero = TRUE, neg_lb = TRUE,
                  alpha = 0.05, n_cl = 2, verbose = TRUE,
                  global = FALSE, pairwise = FALSE, dunnet = FALSE, trend = FALSE,
                  iter_control = list(tol = 1e-2, max_iter = 20, 
                                      verbose = TRUE),
                  em_control = list(tol = 1e-5, max_iter = 100),
                  lme_control = lme4::lmerControl(),
                  mdfdr_control = list(fwer_ctrl_method = "fdr", B = 100))

Checking the input data type ...

The input data is of type: matrix

The imported data is in a generic 'matrix'/'data.frame' format.

PASS

Checking the sample metadata ...

The specified variables in the formula: memdx

The available variables in the sample metadata: memdx, cog_visit_date, visit_type, sex, host_age, host_weight, first_diet, age

PASS

Checking other arguments ...

The number of groups of interest is: 2

Warning message:
“The group variable has < 3 categories 
The multi-group comparisons (global/pairwise/dunnet/trend) will be deactivated”
The sample size per group is: nc = 22, mci = 7

PASS

Warning message in log(O1):
“NaNs produced”
Obtaining initial estimates ...

ML iteration = 1, epsilon = 0.17

ML iteration = 2, epsilon = 1.4

ML iteration = 3, epsilon = 0.094

ML iteration = 4, epsilon = 0.0066

Estimating sample-specific biases ...

Warning message in log(O2):
“NaNs produced”
Warning message in sqrt(sweep(var_hat, 2, var_delta, "*")):
“NaNs produced”
Sensitivit

In [109]:
res_prim = output_human$res
write.table(res_prim,"metagenomics/human_BEAM/ancombbc_primaryreshuman_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)

In [73]:
res_prim = output_lighttrf2$res
write.table(res_prim,"metagenomics/ancombc/ancombbc_primaryreslighttrf2_B10.txt",sep = "\t",row.names = FALSE,quote=FALSE)